# Notebook 20 — multi-day statistical confirmation

Run All launches the full experiment: **56 fresh paired comparisons of ViT and the optimized kernel with an initial local weight of 50%**, training the whole JEPA system. It uses physical **GPU 1 only**, isolated with `CUDA_VISIBLE_DEVICES=1`; the training code sees logical `cuda:0`.

**Runtime:** approximately 60 hours, including 56 hours of training. The original 72-hour deadline never resets. Each model receives 1,800 active wall seconds, at 256px in FP32 with an effective batch of 128. Training ends by hour 64, development readouts by hour 70, and the official readout by hour 72.

**Inputs:** frozen Notebook 19 model definitions and Food101 TRAIN/DEV data. Before training, reserve a fresh 5,050-image official subset. Open its pixels only after all 112 models and TRAIN-fitted classifiers are frozen. The candidate, sample size, primary metric and decision rule are fixed before training. There is no optional stopping or seed replacement; statistical significance is not guaranteed.

**Outputs:** `artifacts/notebook_20/<RUN_ID>/report.html`, paired effects and the statistical result. Full-state checkpoints and manifests live in `checkpoints/<RUN_ID>/`. Retain the latest two valid checkpoints and the best, saving at epoch boundaries and every 50 updates.

**Resume:** use a fresh kernel, the same `RUN_ID`, frozen code and options, then Run All. Existing sessions are reused. A committed READY timing journal permits exact continuation with the original deadline. An interrupted IN_FLIGHT journal or invalid checkpoint remains UNKNOWN; the notebook never silently restarts or forgives unmeasured training time. A terminal result is preserved for inspection.

The detached coordinator and guard continue independently of the chat. The guard writes a heartbeat every 10 seconds and checks integrity hourly. Attach with `tmux attach -t fmm20-multiday-confirmation-001`; detach with Ctrl-b, then d.

Storage is the persistent boot disk; instance-deletion policy remains unverified. Copy final/best checkpoints off-instance and verify hashes before deleting the instance.


In [ ]:
import os,json,types
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']='1'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import nbformat
ROOT=Path('/home/benjamin/workspace/ahr_go/fmm_jeppa')
RUN_ID=os.environ.get('FMM20_RUN_ID','multiday-confirmation-001')
PROFILE=os.environ.get('FMM20_PROFILE','full')
P=ROOT/'experiments/20_multiday_confirmation/workflow.ipynb'
W=types.ModuleType('fmm20_main');W.__file__=str(P)
for cell in nbformat.read(P,4).cells:
    if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(P),'exec'),W.__dict__)


In [ ]:
LAUNCH=W.launch_study(ROOT,RUN_ID,PROFILE)
print(json.dumps(LAUNCH,indent=2,default=str))

In [ ]:
A=ROOT/'artifacts/notebook_20'/RUN_ID
for name in ('study_status.json','guard_heartbeat.json'):
    p=A/name
    if p.exists():
        value=json.loads(p.read_text());print(name,{k:v for k,v in value.items()if k!='training_progress'})